# Assignment 16: AI-Text Detection Robustness

**Important:** `data/sample_data.csv` is illustrative smoke-test data only. Do not report its scores as research results. Replace it with a provenance-documented, topic-matched dataset before drawing conclusions.

Research question: How does light editing/paraphrasing affect AI-origin detection?

This notebook provides dataset checks and the experiment workflow. Use group-wise splitting by `topic_id` so related passages and edited variants cannot leak across train/test partitions.

In [ ]:
from pathlib import Path
import pandas as pd
from src.preprocessing import load_and_validate, word_count

DATA_PATH = Path("../data/sample_data.csv")
df = load_and_validate(DATA_PATH)
df["word_count"] = df["text"].map(word_count)
print("Rows:", len(df))
print("Topics:", df["topic_id"].nunique())
display(df[["passage_id", "topic_id", "condition", "label", "word_count"]])
print("\nCondition counts:")
display(df["condition"].value_counts().rename_axis("condition").to_frame("n"))

## Dataset quality checks

Before running real experiments, verify provenance, topic coverage, condition balance, and passage lengths. The demo dataset is intentionally too small and is not valid evidence.

In [ ]:
# Basic checks to repeat on the real dataset
assert df["passage_id"].is_unique
assert set(df["condition"]).issubset({"human", "ai_original", "ai_edited"})
assert (df.loc[df["condition"].eq("human"), "label"] == 0).all()
assert (df.loc[df["condition"].isin(["ai_original", "ai_edited"]), "label"] == 1).all()
display(df.groupby(["topic_id", "condition"]).size().unstack(fill_value=0))

## Run the main experiment

After replacing the demo CSV with validated research data at `data/your_dataset.csv`, run from the repository root:

```bash
python -m src.model --data data/your_dataset.csv --out results --seed 42
```

The script saves `results.csv`, `predictions.csv`, and `split_manifest.json`. It trains TF-IDF + Logistic Regression and TF-IDF + Linear SVM, with topic-group holdout. Review the topic manifest and verify that all conditions are adequately represented in the test topics before interpreting metrics.

## Interpretation checklist

- Compare the same held-out topics for human vs original AI and human vs edited AI.
- Report false-positive rate explicitly: FP / (FP + TN).
- Report the change in F1 and AI recall after editing.
- Inspect false positives and false negatives qualitatively.
- Do not infer authorship from a detector prediction; report dataset and domain limitations.
- Never present demo-data scores as project findings.